# Cartopy BlueMarble Usage Examples

This notebook demonstrates how to use NASA BlueMarble satellite imagery as backgrounds in Cartopy maps.

## Prerequisites

Before running this notebook:

1. **Download images:**
   ```bash
   cartopy-bg download --profile dev
   ```

2. **Generate images.json:**
   ```bash
   cartopy-bg generate
   ```

3. **Run this notebook:**
   ```bash
   just example-notebook
   ```

   Or manually:
   ```bash
   uv run --with jupyter --with cartopy --with matplotlib --with ipywidgets jupyter notebook examples/cartopy_usage_examples.ipynb
   ```

In [ ]:
# Import required libraries
import os
import sys
import cartopy.crs as ccrs
import cartopy.mpl.geoaxes as geoaxes
import matplotlib.pyplot as plt
from pathlib import Path

# Configure matplotlib for better display in notebooks
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Find project root and images.json
def find_project_root():
    """Find the project root directory containing images.json."""
    current = Path.cwd()
    
    # Try current directory first
    if (current / 'images.json').exists():
        return current
    
    # Try parent directory
    if (current.parent / 'images.json').exists():
        return current.parent
    
    # Try going up until we find it or hit root
    for parent in current.parents:
        if (parent / 'images.json').exists():
            return parent
    
    return None

# Find and change to project root
project_root = find_project_root()

if project_root is None:
    print("⚠️  ERROR: images.json not found!")
    print("")
    print("Please run the following commands first:")
    print("  1. cartopy-bg download --profile dev")
    print("  2. cartopy-bg generate")
    print("")
    print(f"Current directory: {Path.cwd()}")
    print(f"Looked in: {Path.cwd()}, {Path.cwd().parent}, and parent directories")
else:
    # Change to project root so relative paths in images.json work
    os.chdir(project_root)
    images_json = project_root / 'images.json'
    
    print(f"✓ Found project root: {project_root}")
    print(f"✓ Found images.json: {images_json}")
    print(f"✓ Changed working directory to: {Path.cwd()}")
    print("")
    
    # Load custom backgrounds
    try:
        geoaxes.read_user_background_images(str(images_json))
        print("✓ Successfully loaded background images!")
        print("")
        print("Available backgrounds:")
        
        # Show what's available
        import json
        with open(images_json) as f:
            config = json.load(f)
        
        for key in sorted(config.keys()):
            if not key.startswith('__'):
                resolutions = [k for k in config[key].keys() if not k.startswith('__')]
                print(f"  - {key}: {', '.join(resolutions)}")
    except Exception as e:
        print(f"⚠️  ERROR loading backgrounds: {e}")
        print("")
        print("Troubleshooting:")
        print("  1. Check that data/ directory exists with downloaded images")
        print("  2. Run: cartopy-bg verify")
        print("  3. Re-generate: cartopy-bg generate")

## Example 1: Basic Global Map

Create a simple global map with BlueMarble January background.

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Add BlueMarble background (uses 'mid' resolution by default)
ax.background_img(name='Blue Marble January', resolution='mid')

# Add coastlines for reference
ax.coastlines(linewidth=0.5, color='white', alpha=0.7)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

plt.title('Global Map with NASA BlueMarble Background (January)', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## Example 2: Regional Map - Europe

Zoom into a specific region (Europe) with a summer background.

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Add BlueMarble July background
ax.background_img(name='Blue Marble July', resolution='mid')

# Set extent to Europe: [lon_min, lon_max, lat_min, lat_max]
ax.set_extent([-15, 40, 35, 72], crs=ccrs.PlateCarree())

# Add features
ax.coastlines(linewidth=0.8, color='white')
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)

plt.title('European Region - July', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## Example 3: Different Map Projections

Compare how BlueMarble looks with different map projections.

In [ ]:
fig = plt.figure(figsize=(16, 10))

# PlateCarree (equirectangular)
ax1 = fig.add_subplot(2, 2, 1, projection=ccrs.PlateCarree())
ax1.background_img(name='Blue Marble April', resolution='mid')
ax1.coastlines(linewidth=0.5, color='white', alpha=0.7)
ax1.set_title('PlateCarree Projection')

# Orthographic (globe view) - North Pole
ax2 = fig.add_subplot(2, 2, 2, projection=ccrs.Orthographic(0, 90))
ax2.background_img(name='Blue Marble April', resolution='mid')
ax2.coastlines(linewidth=0.5, color='white', alpha=0.7)
ax2.set_title('Orthographic - North Pole')

# Mollweide (elliptical)
ax3 = fig.add_subplot(2, 2, 3, projection=ccrs.Mollweide())
ax3.background_img(name='Blue Marble April', resolution='mid')
ax3.coastlines(linewidth=0.5, color='white', alpha=0.7)
ax3.set_title('Mollweide Projection')

# Robinson
ax4 = fig.add_subplot(2, 2, 4, projection=ccrs.Robinson())
ax4.background_img(name='Blue Marble April', resolution='mid')
ax4.coastlines(linewidth=0.5, color='white', alpha=0.7)
ax4.set_title('Robinson Projection')

plt.suptitle('BlueMarble April - Different Projections', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## Example 4: Seasonal Changes

Visualize how Earth's appearance changes across seasons.

In [ ]:
fig = plt.figure(figsize=(16, 8))

months = ['January', 'April', 'July', 'October']

for idx, month in enumerate(months, 1):
    ax = fig.add_subplot(2, 2, idx, projection=ccrs.PlateCarree())
    ax.background_img(name=f'Blue Marble {month}', resolution='mid')
    
    # Focus on North America to see seasonal changes
    ax.set_extent([-130, -60, 25, 50], crs=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.5, color='white', alpha=0.8)
    
    ax.set_title(month, fontsize=12)

plt.suptitle('Seasonal Changes - North America', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## Example 5: Resolution Comparison

Compare different resolution levels (low, mid, high).

In [ ]:
fig = plt.figure(figsize=(16, 5))

resolutions = [('low', '720x360'), ('mid', '1440x720'), ('high', '3600x1800')]

for idx, (res_name, res_size) in enumerate(resolutions, 1):
    ax = fig.add_subplot(1, 3, idx, projection=ccrs.PlateCarree())
    
    try:
        ax.background_img(name='Blue Marble January', resolution=res_name)
        
        # Zoom into Mediterranean to see detail differences
        ax.set_extent([-10, 30, 35, 60], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.5, color='white')
        
        ax.set_title(f'{res_name.capitalize()} ({res_size})', fontsize=11)
    except Exception as e:
        # If resolution not available
        ax.text(0.5, 0.5, f'{res_name} resolution\nnot available',
                ha='center', va='center', transform=ax.transAxes, fontsize=10, color='red')
        ax.set_title(f'{res_name.capitalize()} ({res_size})', fontsize=11)

plt.suptitle('Resolution Comparison - Mediterranean Region', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## Interactive Example: Choose Your Own

Create an interactive map where you can select the month and resolution.

In [ ]:
from ipywidgets import interact, Dropdown

def plot_bluemarble(month='January', resolution='mid'):
    """Interactive function to plot BlueMarble with selected parameters."""
    fig = plt.figure(figsize=(14, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    
    try:
        ax.background_img(name=f'Blue Marble {month}', resolution=resolution)
        ax.coastlines(linewidth=0.5, color='white', alpha=0.7)
        
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        
        plt.title(f'Blue Marble - {month} ({resolution} resolution)', fontsize=14, pad=15)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error loading image:\n{str(e)}',
                ha='center', va='center', transform=ax.transAxes, fontsize=12, color='red')
        plt.title(f'Blue Marble - {month} ({resolution} resolution) - NOT AVAILABLE', 
                  fontsize=14, pad=15, color='red')
    
    plt.tight_layout()
    plt.show()

# Create interactive widgets
months = ['January', 'February', 'March', 'April', 'May', 'June',
          'July', 'August', 'September', 'October', 'November', 'December']

interact(plot_bluemarble,
         month=Dropdown(options=months, value='January', description='Month:'),
         resolution=Dropdown(options=['low', 'mid', 'high'], value='mid', description='Resolution:'));

## Next Steps

### Verify Your Downloads
Check that all images are valid:
```bash
cartopy-bg verify
```

### Download More Data
Get additional datasets or resolutions:
```bash
# Download high resolution
cartopy-bg download -r high

# Download all datasets
cartopy-bg download -d all --profile prod
```

### Explore More
- Try different map projections: `ccrs.Mercator()`, `ccrs.LambertConformal()`, etc.
- Add your own data layers (points, lines, polygons)
- Combine multiple months to create animations
- Use different datasets if you've downloaded `bluemarble-tb` (topography/bathymetry)

### Resources
- [Cartopy Documentation](https://scitools.org.uk/cartopy/docs/latest/)
- [NASA Visible Earth](https://visibleearth.nasa.gov/)
- [Project README](../README.md)